<h1>Importing Dependencies</h1>

In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
import seaborn as sns

<h1>Data Collection and Preprocessing</h1>

In [14]:
breast_cancer=pd.read_csv("breast_cancer_enhanced_dataset.csv")
print(breast_cancer.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5500 entries, 0 to 5499
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id                            5500 non-null   float64
 1   diagnosis                     5500 non-null   object 
 2   radius_mean                   5500 non-null   float64
 3   texture_mean                  5500 non-null   float64
 4   perimeter_mean                5500 non-null   float64
 5   area_mean                     5500 non-null   float64
 6   smoothness_mean               5500 non-null   float64
 7   compactness_mean              5500 non-null   float64
 8   concavity_mean                5500 non-null   float64
 9   concave points_mean           5500 non-null   float64
 10  shape_irregularity            5500 non-null   float64
 11  border_complexity             5500 non-null   float64
 12  tumor_aggressiveness          5500 non-null   float64
 13  rad

In [15]:
print(breast_cancer.iloc[0:5,:])

             id diagnosis  radius_mean  texture_mean  perimeter_mean  \
0  5.683796e+06         B    11.829858     21.726166       75.154378   
1 -6.253379e+06         B    10.991150     17.103260       71.798929   
2  4.213892e+06         M    21.433519     15.092437      142.753006   
3 -2.986069e+06         B    11.700452     14.872127       74.154481   
4  6.469594e+05         B    13.259377     17.212990       83.621014   

     area_mean  smoothness_mean  compactness_mean  concavity_mean  \
0   435.022394         0.087089          0.050717        0.015868   
1   381.386295         0.089339          0.109498        0.097344   
2  1392.399890         0.099557          0.152079        0.193372   
3   404.112556         0.101291          0.077563        0.043749   
4   521.124238         0.072905          0.043312        0.046983   

   concave points_mean  shape_irregularity  border_complexity  \
0             0.011641            0.078226           0.000185   
1             0.035629

In [16]:
print(breast_cancer.shape)

(5500, 17)


In [17]:
print(breast_cancer.isnull().sum())

id                              0
diagnosis                       0
radius_mean                     0
texture_mean                    0
perimeter_mean                  0
area_mean                       0
smoothness_mean                 0
compactness_mean                0
concavity_mean                  0
concave points_mean             0
shape_irregularity              0
border_complexity               0
tumor_aggressiveness            0
radius_texture_interaction      0
radius_concavity_interaction    0
concavity_density               0
malignancy_risk_score           0
dtype: int64


In [18]:
breast_cancer["diagnosis"]=np.where(breast_cancer["diagnosis"]=='B', 0,1)

In [19]:
#knowing about our data
breast_cancer.describe()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,shape_irregularity,border_complexity,tumor_aggressiveness,radius_texture_interaction,radius_concavity_interaction,concavity_density,malignancy_risk_score
count,5.500000e+03,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000,5500.000000
mean,2.705450e+07,0.369091,14.095290,19.317068,91.749574,651.332914,0.096027,0.103069,0.086652,0.048172,0.237894,0.006983,0.079298,276.771030,1.408473,0.000129,29.831833
std,1.135108e+08,0.482602,3.484868,4.350629,24.012506,346.027394,0.013878,0.052939,0.078752,0.038531,0.164068,0.011284,0.054689,107.515404,1.605029,0.000117,9.282654
min,-1.637568e+07,0.000000,6.817646,9.522736,42.639188,123.928240,0.052200,0.015551,-0.006232,-0.002668,0.017291,-0.000020,0.005764,92.849788,-0.064863,-0.000023,12.370635
25%,-4.464305e+05,0.000000,11.679531,16.196326,75.119335,420.093960,0.085863,0.063526,0.028775,0.019967,0.113784,0.000600,0.037928,199.843614,0.342645,0.000061,23.293325
50%,2.982231e+06,0.000000,13.339209,18.817023,86.222090,549.472961,0.095362,0.090793,0.058934,0.032845,0.190299,0.001889,0.063433,246.009655,0.745327,0.000107,26.953460
75%,8.362453e+06,1.000000,15.760529,21.841027,103.688058,776.638240,0.105008,0.129659,0.125716,0.072017,0.327721,0.008902,0.109240,338.834746,1.992114,0.000169,35.026766
max,9.235889e+08,1.000000,28.341004,39.442971,188.983008,2507.982355,0.164055,0.348835,0.429627,0.202662,0.919553,0.086579,0.306518,725.185848,10.439435,0.001431,67.030216


In [20]:
#checking the distributing of target value
print(breast_cancer["diagnosis"].value_counts())
# 0 represents benign
# 1 represents malignant

diagnosis
0    3470
1    2030
Name: count, dtype: int64


In [21]:
print(breast_cancer.groupby("diagnosis").mean())

                     id  radius_mean  texture_mean  perimeter_mean  \
diagnosis                                                            
0          2.545944e+07    12.147692     18.000635       78.109806   
1          2.978104e+07    17.424433     21.567326      115.064843   

            area_mean  smoothness_mean  compactness_mean  concavity_mean  \
diagnosis                                                                  
0          463.111490         0.092341          0.079051        0.044774   
1          973.071014         0.102327          0.144124        0.158238   

           concave points_mean  shape_irregularity  border_complexity  \
diagnosis                                                               
0                     0.025445            0.149269           0.001625   
1                     0.087022            0.389384           0.016141   

           tumor_aggressiveness  radius_texture_interaction  \
diagnosis                                                 

<h1>Checking For Outliers</h1>

In [22]:
for col in breast_cancer.columns:
    Q1 = breast_cancer[col].quantile(0.25)
    Q3 = breast_cancer[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    outliers = ((breast_cancer[col] < lower) | (breast_cancer[col] > upper)).sum()
    print(col, round(100*outliers/len(X), 2), "%")

id 13.75 %
diagnosis 0.0 %
radius_mean 2.27 %
texture_mean 1.44 %
perimeter_mean 2.4 %
area_mean 4.78 %
smoothness_mean 1.07 %
compactness_mean 2.82 %
concavity_mean 3.33 %
concave points_mean 1.96 %
shape_irregularity 2.47 %
border_complexity 8.75 %
tumor_aggressiveness 2.47 %
radius_texture_interaction 2.29 %
radius_concavity_interaction 5.09 %
concavity_density 4.45 %
malignancy_risk_score 2.04 %


<h1>Seperating inputs and outputs</h1>

In [23]:
X=breast_cancer.drop(columns=["diagnosis","id", "malignancy_risk_score", "tumor_aggressiveness"])
Y=breast_cancer["diagnosis"]

<h1>Splitting the data into training and testing data</h1>

In [24]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=12)

In [25]:
print("shape of training inputs: ",X_train.shape)
print("shape of training outputs: ",X_test.shape)

shape of training inputs:  (4400, 13)
shape of training outputs:  (1100, 13)


<h1>Scaling the data</h1>

In [26]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

<h1>Finding Best Model for our project</h1>

In [27]:
def bestmodel(models,X,Y):
    for model in models:
        scaler=StandardScaler()
        pipeline=Pipeline([('scaler', scaler), ("model", model)])
        print("Expected Accuracy of model",type(model).__name__," is ",cross_val_score(pipeline,X,Y,scoring="accuracy",cv=10, n_jobs=-1).mean())
models=[LogisticRegression(),DecisionTreeClassifier(),RandomForestClassifier(), SVC()]
bestmodel(models,X,Y)

Expected Accuracy of model LogisticRegression  is  0.9401818181818182
Expected Accuracy of model DecisionTreeClassifier  is  0.9905454545454544
Expected Accuracy of model RandomForestClassifier  is  0.9974545454545455
Expected Accuracy of model SVC  is  0.9685454545454546


<h1>Training Model</h1>

In [28]:
# using random forest
rfc=RandomForestClassifier()
rfc.fit(X_train,Y_train)

RandomForestClassifier()

<h1>Evaluating Model</h1>

In [29]:
Y_pred=rfc.predict(X_test)
print("Accuracy of our model: ",accuracy_score(Y_test,Y_pred))

Accuracy of our model:  0.9990909090909091


In [30]:
from sklearn.metrics import classification_report

In [31]:
classification_report(Y_test,Y_pred, output_dict=True)

{'0': {'precision': 0.9985569985569985,
  'recall': 1.0,
  'f1-score': 0.9992779783393502,
  'support': 692.0},
 '1': {'precision': 1.0,
  'recall': 0.9975490196078431,
  'f1-score': 0.9987730061349693,
  'support': 408.0},
 'accuracy': 0.9990909090909091,
 'macro avg': {'precision': 0.9992784992784993,
  'recall': 0.9987745098039216,
  'f1-score': 0.9990254922371598,
  'support': 1100.0},
 'weighted avg': {'precision': 0.9990922209104027,
  'recall': 0.9990909090909091,
  'f1-score': 0.9990906795580891,
  'support': 1100.0}}

In [32]:
rfc.feature_importances_

array([0.02663271, 0.03443139, 0.09318614, 0.07420723, 0.02380921,
       0.0198764 , 0.05330761, 0.16483751, 0.11675017, 0.14167282,
       0.07375512, 0.15788719, 0.01964651])

In [33]:
print(breast_cancer.duplicated().sum())

0


In [34]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=12)

In [35]:
model=Pipeline([("scaler", StandardScaler()), ("model", rfc)])

In [36]:
model.fit(X_train,Y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', RandomForestClassifier())])

In [37]:
Y_pred=model.predict(X_test)

In [38]:
print("Accuracy of our model: ",accuracy_score(Y_test,Y_pred))

Accuracy of our model:  0.9990909090909091


In [39]:
import pickle
with open("model.pkl","wb") as file:
    pickle.dump(model,file)

In [40]:
model.predict(X.iloc[0,:].values.reshape(1,-1))

c:\Users\shagu\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


array([0])

In [41]:
X

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,shape_irregularity,border_complexity,radius_texture_interaction,radius_concavity_interaction,concavity_density
0,11.829858,21.726166,75.154378,435.022394,0.087089,0.050717,0.015868,0.011641,0.078226,0.000185,257.017451,0.187718,0.000036
1,10.991150,17.103260,71.798929,381.386295,0.089339,0.109498,0.097344,0.035629,0.242470,0.003468,187.984489,1.069921,0.000255
2,21.433519,15.092437,142.753006,1392.399890,0.099557,0.152079,0.193372,0.126922,0.472373,0.024543,323.484034,4.144647,0.000139
3,11.700452,14.872127,74.154481,404.112556,0.101291,0.077563,0.043749,0.028747,0.150059,0.001258,174.010599,0.511885,0.000108
4,13.259377,17.212990,83.621014,521.124238,0.072905,0.043312,0.046983,0.010096,0.100391,0.000474,228.233523,0.622960,0.000090
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5495,11.688492,18.217548,75.277156,429.895271,0.113777,0.104274,0.069515,0.034620,0.208409,0.002407,212.935657,0.812528,0.000162
5496,13.831066,19.454900,88.204429,580.072530,0.087240,0.061584,0.012322,0.022518,0.096424,0.000277,269.082007,0.170427,0.000021
5497,19.403923,18.152520,127.631265,1129.627896,0.104142,0.144262,0.164108,0.094511,0.402881,0.015510,352.230108,3.184337,0.000145
5498,20.946319,25.233666,142.498720,1340.103257,0.109454,0.226277,0.316075,0.148336,0.690688,0.046885,528.552405,6.620608,0.000236


In [42]:
breast_cancer

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,shape_irregularity,border_complexity,tumor_aggressiveness,radius_texture_interaction,radius_concavity_interaction,concavity_density,malignancy_risk_score
0,5.683796e+06,0,11.829858,21.726166,75.154378,435.022394,0.087089,0.050717,0.015868,0.011641,0.078226,0.000185,0.026075,257.017451,0.187718,0.000036,22.433798
1,-6.253379e+06,0,10.991150,17.103260,71.798929,381.386295,0.089339,0.109498,0.097344,0.035629,0.242470,0.003468,0.080823,187.984489,1.069921,0.000255,24.021839
2,4.213892e+06,1,21.433519,15.092437,142.753006,1392.399890,0.099557,0.152079,0.193372,0.126922,0.472373,0.024543,0.157458,323.484034,4.144647,0.000139,49.053992
3,-2.986069e+06,0,11.700452,14.872127,74.154481,404.112556,0.101291,0.077563,0.043749,0.028747,0.150059,0.001258,0.050020,174.010599,0.511885,0.000108,23.276145
4,6.469594e+05,0,13.259377,17.212990,83.621014,521.124238,0.072905,0.043312,0.046983,0.010096,0.100391,0.000474,0.033464,228.233523,0.622960,0.000090,25.647074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5495,2.909607e+06,0,11.688492,18.217548,75.277156,429.895271,0.113777,0.104274,0.069515,0.034620,0.208409,0.002407,0.069470,212.935657,0.812528,0.000162,24.344788
5496,4.399236e+05,0,13.831066,19.454900,88.204429,580.072530,0.087240,0.061584,0.012322,0.022518,0.096424,0.000277,0.032141,269.082007,0.170427,0.000021,26.379882
5497,-4.461865e+06,1,19.403923,18.152520,127.631265,1129.627896,0.104142,0.144262,0.164108,0.094511,0.402881,0.015510,0.134294,352.230108,3.184337,0.000145,43.224263
5498,7.519916e+06,1,20.946319,25.233666,142.498720,1340.103257,0.109454,0.226277,0.316075,0.148336,0.690688,0.046885,0.230229,528.552405,6.620608,0.000236,52.471548
